# Lab 04: Parameter-Efficient Fine-Tuning (LoRA) with `Qwen2.5-0.5B-Instruct`

**Module:** PUSL 3205 — Large Language Models and Generative AI
**Lab 04:** Fine-Tuning an LLM using LoRA / QLoRA
**Author:** Kumarage Isiwara Didulani

---

## Overview
This notebook demonstrates supervised fine-tuning (SFT) using **Low-Rank Adaptation (LoRA)** on `Qwen/Qwen2.5-0.5B-Instruct` for a custom university campus shop domain (**Campus Essentials**). Using a 30-pair domain Q&A dataset and Hugging Face's `peft` and `trl` (`SFTTrainer`) libraries, the model adapts to domain-specific policies, pricing, and inventory while training less than `0.9%` of its total parameters.

### Step 1: Install Required Python Libraries

**Question:**
> *What packages are required to set up an efficient parameter-efficient fine-tuning (PEFT) and supervised fine-tuning (SFT) environment for large language models[cite: 115]?*

**Explanation:**
* **`transformers` & `datasets`**: Core Hugging Face libraries for loading pre-trained models, tokenizers, and custom datasets[cite: 73, 115].
* **`peft` & `trl`**: Parameter-Efficient Fine-Tuning library (for LoRA) and Transformer Reinforcement Learning library (for `SFTTrainer`)[cite: 73, 115].
* **`accelerate` & `bitsandbytes`**: Optimizes memory and enables low-precision training (quantization)[cite: 73, 115].

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes "torchao>=0.16.0"

### Step 2: Verify GPU Hardware & Memory

**Question:**
> *How do we verify whether a GPU is available, and why is hardware acceleration necessary for fine-tuning LLMs[cite: 73, 115]?*

**Explanation:**
* **Hardware Acceleration**: Fine-tuning millions of neural network weights via backpropagation requires massive parallel matrix multiplication, making GPUs (such as the Tesla T4) essential[cite: 73, 100, 115].
* **Memory Check**: Inspects total VRAM availability to ensure the model and training batch sizes fit within hardware limits[cite: 73, 115].

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print("GPU Memory:", round(gpu.total_memory / 1024**3, 2), "GB")
else:
    print("No GPU available")

### Step 3: Create a Domain-Specific Dataset

**Question:**
> *How do we construct a custom question-and-answer dataset for domain adaptation (e.g., a university campus shop)[cite: 73, 115]?*

**Explanation:**
* **Domain Adaptation**: Pre-trained models understand general language[cite: 78] but lack specific internal knowledge about local services (like "Campus Essentials")[cite: 73].
* **Structured Q&A**: We define clean prompt-response pairs using the Hugging Face `Dataset` API[cite: 73, 99].

In [ ]:
from datasets import Dataset

data = [
    {"question": "What products does Campus Essentials sell?", "answer": "Campus Essentials sells notebooks, pens, textbooks, USB drives, water bottles, and university merchandise."},
    {"question": "How much does a notebook cost?", "answer": "A standard notebook costs LKR 350."},
    {"question": "Do you sell computer science textbooks?", "answer": "Yes. Selected computer science textbooks are available, subject to stock availability."},
    {"question": "What time does the shop open?", "answer": "The shop opens at 8:30 AM from Monday to Friday."},
    {"question": "What time does Campus Essentials close?", "answer": "The shop closes at 5:30 PM on weekdays."},
    {"question": "Is Campus Essentials open on weekends?", "answer": "No. The shop is closed on Saturdays, Sundays, and university public holidays."},
    {"question": "Can I return a damaged product?", "answer": "Yes. Customers can request an exchange for a damaged product by presenting the receipt within seven days of purchase."},
    {"question": "Do you offer cash refunds for returned items?", "answer": "No. Campus Essentials offers product exchanges or store credit rather than cash refunds."},
    {"question": "Do you sell USB drives?", "answer": "Yes. USB drives are available in 16 GB, 32 GB, and 64 GB storage capacities."},
    {"question": "How much is a 32 GB USB drive?", "answer": "A 32 GB USB drive costs LKR 1,850."},
    {"question": "Can I pay by card?", "answer": "Yes. The shop accepts cash, debit cards, credit cards, and QR mobile payments."},
    {"question": "Are university-branded T-shirts available?", "answer": "Yes. University-branded T-shirts are available in sizes S, M, L, and XL."},
    {"question": "How much does a university T-shirt cost?", "answer": "A university-branded T-shirt costs LKR 2,200."},
    {"question": "Can I reserve a textbook?", "answer": "Yes. Customers can request a textbook reservation by contacting the shop. Reservations depend on stock availability."},
    {"question": "Where is Campus Essentials located?", "answer": "Campus Essentials is located on the ground floor beside the main university library."},
    {"question": "Do you sell scientific calculators?", "answer": "Yes. Casio scientific calculators suitable for engineering and data science exams are in stock."},
    {"question": "Do you provide printing or binding services?", "answer": "Yes. We offer black-and-white printing, color printing, and spiral binding for project reports."},
    {"question": "How much does spiral binding cost?", "answer": "Spiral binding costs LKR 250 per document, including front and back covers."},
    {"question": "Do you sell laptop chargers or HDMI cables?", "answer": "Yes. We stock universal Type-C laptop chargers, HDMI cables, and wireless mice."},
    {"question": "Is there a student discount available?", "answer": "Yes. Students receive a 10% discount on stationery bundles when presenting a valid student ID."},
    {"question": "Do you sell reusable water bottles?", "answer": "Yes. BPA-free reusable sports bottles and insulated flasks are available."},
    {"question": "Can I order items online for pickup?", "answer": "Yes. Students can place same-day pickup orders through the Campus Essentials portal before 3:00 PM."},
    {"question": "Do you sell A4 drawing pads and lab logbooks?", "answer": "Yes. Hardcover lab logbooks and A4 drawing pads are stocked throughout the semester."},
    {"question": "How much does a hardcover lab logbook cost?", "answer": "A hardcover lab logbook costs LKR 650."},
    {"question": "Do you sell sticky notes and highlighters?", "answer": "Yes. Multi-color sticky notes, index tabs, and pastel highlighter packs are available."},
    {"question": "Can faculty members purchase bulk office supplies?", "answer": "Yes. Academic departments and faculty members can place bulk orders with official purchase vouchers."},
    {"question": "Do you sell earphones or headphones?", "answer": "Yes. Basic wired earphones and budget over-ear headphones are available."},
    {"question": "What happens if an item is out of stock?", "answer": "You can leave your student email at the counter to receive a restock notification within three working days."},
    {"question": "Do you sell umbrella or rain gear?", "answer": "Yes. Compact foldable umbrellas are available at the front counter."},
    {"question": "How can I contact Campus Essentials?", "answer": "You can reach Campus Essentials at campus.essentials@university.edu or visit the counter beside the library."}
]

print("Number of examples:", len(data))
dataset = Dataset.from_list(data)

### Step 4: Import Base Model and Tokenizer

**Question:**
> *How do we load an instruction-tuned causal language model (such as `Qwen/Qwen2.5-0.5B-Instruct`) and configure its tokenizer for CPU execution[cite: 73, 115]?*

**Explanation:**
* **Model Loading**: Imports `AutoModelForCausalLM` using standard 32-bit floating-point precision (`torch.float32`) since CUDA is not present[cite: 73].
* **Padding Token**: Ensures the tokenizer has a valid padding token configured by setting it equal to the end-of-sequence (`eos_token`)[cite: 73].

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Force float32 for CPU compatibility
dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype
)

print("Model loaded successfully on CPU!")
print("Parameters:", f"{model.num_parameters():,}")

### Step 5: Construct Chat Template Formatting

**Question:**
> *How do we format raw text pairs into standardized model-specific chat templates for instruction tuning[cite: 73, 81]?*

**Explanation:**
* **Chat Templates**: Wraps user questions and assistant answers using role markers so the model learns conversational structure[cite: 73, 81].

In [ ]:
def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }

train_dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

print(train_dataset[0]["text"])

### Step 6: Parameter-Efficient Fine-Tuning (LoRA) Configuration

**Question:**
> *How do we configure Low-Rank Adaptation (LoRA) to update only a tiny fraction of model weights[cite: 101, 102]?*

**Explanation:**
* **LoRA Setup**: Inserts low-rank decomposition matrices into attention projections, reducing the number of parameters trained[cite: 101, 102].

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### Step 7: Train the Model Using SFTTrainer (CPU Version)

**Question:**
> *How do we execute supervised fine-tuning using Hugging Face's TRL library (`SFTTrainer`) on a CPU-only local environment[cite: 73, 98]?*

**Explanation:**
* **CPU Training Configuration**: We set `fp16=False` and use standard 32-bit floating-point precision so the training loop runs correctly on your CPU without throwing CUDA errors[cite: 73].
* **Execution**: `trainer.train()` updates our LoRA adapter weights using our custom campus shop dataset[cite: 73].

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig

print("Configuring SFTTrainer for CPU execution...")

training_args = SFTConfig(
    output_dir="./uni_fundamentals_lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,  # Lower batch size for CPU efficiency
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=2,
    save_strategy="no",  # Skip saving checkpoints during quick CPU run to save time
    fp16=False,          # Must be False for CPU
    bf16=False,
    gradient_checkpointing=False,
    max_length=256,
    dataset_text_field="text",
    report_to="none",
    seed=42
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer
)

print("Starting training loop...")
trainer.train()
print("Training completed successfully!")

### Step 8: Save the Model Adapters

**Question:**
> *How do we serialize and save our trained LoRA adapter weights for future inference and deployment[cite: 73]?*

**Explanation:**
* **Adapter Persistence**: Saves only the lightweight LoRA parameter weights and tokenizer settings rather than the entire base model[cite: 73].

In [ ]:
output_dir = "./uni_fundamentals_lora/final_adapter"

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("LoRA adapter saved successfully to:", output_dir)

### Step 9: Test Unseen Questions on Base vs. Fine-Tuned Models

**Question:**
> *How do we evaluate and compare responses from the original base model versus the fine-tuned model using questions absent from the training dataset[cite: 73]?*

**Explanation:**
* **Inference Pipeline**: Loads the base model alongside the saved LoRA adapter using `PeftModel.from_pretrained` and generates text responses for qualitative comparison[cite: 73].

In [ ]:
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

test_questions = [
    "Do you sell laptop chargers?",
    "Can I buy stationery and textbooks from the same shop?",
    "Can I get a cash refund if I change my mind about a notebook?",
    "Where should I go on campus to get my final-year report spiral bound?"
]

def generate_answer(target_model, question):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(target_model.device)

    target_model.eval()
    with torch.no_grad():
        outputs = target_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Clear training model from memory
if "model" in globals():
    del model
gc.collect()

# 1. Load Original Base Model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
)

print("==========================================")
print("ORIGINAL BASE MODEL vs. FINE-TUNED MODEL")
print("==========================================")

base_answers = [generate_answer(base_model, q) for q in test_questions]

# 2. Attach Fine-Tuned LoRA Adapter
fine_tuned_model = PeftModel.from_pretrained(base_model, "../uni_fundamentals_lora/final_adapter")
ft_answers = [generate_answer(fine_tuned_model, q) for q in test_questions]

for idx, question in enumerate(test_questions):
    print(f"\nQ{idx+1}: {question}")
    print(f"  [Base Model]       : {base_answers[idx]}")
    print(f"  [Fine-Tuned LoRA]  : {ft_answers[idx]}")